# Submission

### Load all other models made and create the final submission csv for that score.

### Import Libraries

In [1]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np

### Load Models

In [2]:
models_dir = Path("../models")

### Baseline model

In [3]:
log_reg = joblib.load(models_dir / "logistic_regression.joblib")
nominal_encoder = joblib.load(models_dir / "nominal_encoder.joblib")
scaler = joblib.load(models_dir / "scaler.joblib")
rating_medians = joblib.load(models_dir / "rating_medians.joblib")

### Load Test Dataset

In [4]:
test_df = pd.read_csv("../data/test.csv")

In [5]:
print("Test shape:", test_df.shape)
display(test_df.head())

Test shape: (299844, 22)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class
0,699635,25,1197,4,4,4,2,3,4,3,...,5,2,1,3,0,0.0,Female,disloyal Customer,Business travel,Eco
1,699636,36,829,5,5,5,5,5,4,5,...,2,4,3,5,0,0.0,Male,Loyal Customer,Business travel,Eco
2,699637,45,472,4,5,5,5,4,4,4,...,3,1,3,4,0,0.0,Male,Loyal Customer,Business travel,Eco
3,699638,45,762,2,2,2,5,3,2,3,...,4,4,4,3,0,0.0,Female,disloyal Customer,Business travel,Business
4,699639,34,594,1,2,1,1,2,1,2,...,4,2,1,2,0,0.0,Male,Loyal Customer,Personal Travel,Eco Plus


### Cleaning Test Data

In [6]:
test_ids = test_df["id"].copy()

In [7]:
X_test = test_df.drop(columns=["id"]).copy()

X_test["Arrival Delay in Minutes"] = (
    X_test["Arrival Delay in Minutes"].fillna(0) # 0 was proven in train_df as median so we fill that here as well
)

In [8]:
print("Test feature shape:", X_test.shape)
print("Missing arrival delays:", X_test["Arrival Delay in Minutes"].isna().sum())

Test feature shape: (299844, 21)
Missing arrival delays: 0


### Encoding

In [9]:
nominal_cols = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

# Transform using the already-fitted encoder
nominal_test_array = nominal_encoder.transform(X_test[nominal_cols])

# Convert encoded values into a DataFrame
nominal_test_df = pd.DataFrame(
    nominal_test_array,
    columns=nominal_encoder.get_feature_names_out(nominal_cols),
    index=X_test.index
)

# Replace original categorical columns with encoded columns
X_test_encoded = (
    X_test.drop(columns=nominal_cols)
    .join(nominal_test_df)
)

print("Shape after encoding:", X_test_encoded.shape)

Shape after encoding: (299844, 26)


#### Ratings

In [10]:

# Rating columns used during training
rating_cols = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness"
]

# Record which ratings were originally unrated
for col in rating_cols:
    X_test_encoded[f"{col}_unrated"] = (X_test_encoded[col] == 0).astype(int)

# Replace unrated zeros with missing values
for col in rating_cols:
    X_test_encoded[col] = X_test_encoded[col].replace(0, np.nan)

# Impute using medians learned from the training data
for col in rating_cols:
    X_test_encoded[col] = X_test_encoded[col].fillna(rating_medians[col])

print("Shape after rating processing:", X_test_encoded.shape)
print("Remaining missing values:", X_test_encoded.isna().sum().sum())


Shape after rating processing: (299844, 39)
Remaining missing values: 0


### Scaling

In [11]:
numeric_cols = ["Age", "Flight Distance",
                "Departure Delay in Minutes", "Arrival Delay in Minutes"]
X_test_encoded[numeric_cols] = scaler.transform(X_test_encoded[numeric_cols])

### Predictions on Test and submission

In [12]:
test_proba = log_reg.predict_proba(X_test_encoded)[:, 1]

# Convert probabilities into binary class predictions
test_pred = log_reg.predict(X_test_encoded)

print("Number of predictions:", len(test_pred))
print("Predictions shape:", test_pred.shape)
print("First 10 predictions:", test_pred[:10])


Number of predictions: 299844
Predictions shape: (299844,)
First 10 predictions: [0 1 1 0 0 1 0 1 1 0]


In [13]:
sample_submission = pd.read_csv(
    "../data/sample_submission.csv"
)

print("Sample submission shape:", sample_submission.shape)
display(sample_submission.head())
print("Submission columns:", sample_submission.columns.tolist())


Sample submission shape: (299844, 2)


,id,satisfaction
0,699635,0.443573
1,699636,0.443573
2,699637,0.443573
3,699638,0.443573
4,699639,0.443573


Submission columns: ['id', 'satisfaction']


In [14]:
# Start with Kaggle's required submission format
submission = sample_submission.copy()

# Insert the test IDs and predicted satisfaction probabilities
submission["id"] = test_ids.to_numpy()
submission["satisfaction"] = test_proba

# Save the CSV
submission_path = "../submissions/logreg_baseline.csv"
submission.to_csv(submission_path, index=False)

# Validate the output
print("Submission shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("IDs match test data:", submission["id"].equals(test_ids.reset_index(drop=True)))
print("Missing values:", submission.isna().sum().sum())
print(submission["satisfaction"].describe())

display(submission.head())


Submission shape: (299844, 2)
Columns: ['id', 'satisfaction']
IDs match test data: True
Missing values: 0
count    299844.000000
mean          0.443918
std           0.394562
min           0.000036
25%           0.043585
50%           0.320558
75%           0.893256
max           0.999724
Name: satisfaction, dtype: float64


,id,satisfaction
0,699635,0.113476
1,699636,0.642442
2,699637,0.505904
3,699638,0.042772
4,699639,0.001079
